# 考古研究日志

**目的**: 记录考古 AI/ML 领域的调研发现，包括数据集、工具、方法论。
**成功标准**: 整理出对我们项目（墓葬分析）有实际参考价值的信息。

In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
from datetime import datetime

print(f"Log entry: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 调研：考古 AI/ML 数据集现状

**核心发现**: 考古领域**没有公认的"标准大数据集"**——这是领域的现实困境。

数据极度碎片化，各团队自建数据集，没有统一标准。以下是我们调研到的信息。

In [ ]:
# 考古 CV 论文中实际引用最多的通用预训练底座

archaeology_pretrained = {
    "ImageNet": {
        "用途": "几乎所有考古 CV 论文都用它做迁移学习的预训练权重",
        "类型": "通用图像分类数据集",
        "规模": "~1400万图片, 1000类",
        "考古场景": "器物分类、墓葬图像识别、遗址照片分析",
    },
    "COCO": {
        "用途": "目标检测任务的预训练底座",
        "引用率": "约 41% 的考古结构检测研究用到",
        "类型": "通用目标检测/分割数据集",
        "规模": "~33万图片, 80类",
        "考古场景": "遗址结构检测、器物定位、航拍目标识别",
    },
}

for name, info in archaeology_pretrained.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in info.items():
        print(f"  {k}: {v}")

In [ ]:
# 考古专用数据集（相对较知名）

archaeology_datasets = {
    "Archaeoscape": {
        "发布": "2024, NeurIPS",
        "类型": "LiDAR 遗址检测",
        "特点": "目前学术引用最新、最正规的考古专用数据集",
        "适用": "地表遗址识别、遥感考古",
        "与我们项目关系": "不直接相关（我们做墓葬分析，非遥感），但标注规范值得参考",
    },
    "Kaggle OpenAI to Z Challenge": {
        "发布": "2024-2025",
        "类型": "考古 AI 竞赛数据",
        "特点": "今年热度最高的考古 AI 竞赛",
        "适用": "遗址发现、卫星图像分析",
        "与我们项目关系": "竞赛思路可参考，但数据类型不同",
    },
}

for name, info in archaeology_datasets.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    for k, v in info.items():
        print(f"  {k}: {v}")

In [ ]:
# 对我们项目的实际策略

strategy = {
    "第一步": {
        "动作": "用 ImageNet/COCO 预训练模型做迁移学习",
        "原因": "考古领域普遍采用，不需要自己从头训练",
        "具体": "PyTorch torchvision.models 预训练权重",
    },
    "第二步": {
        "动作": "自己标注小规模领域数据",
        "工具推荐": "Roboflow（在线标注平台，支持导出多种格式）",
        "规模": "100-500 张图片即可开始微调",
        "标注内容": "墓葬形制、器物类型、纹饰类别",
    },
    "第三步": {
        "动作": "参考 Archaeoscape 的标注规范",
        "原因": "2024年最新、最正规的考古数据集标注标准",
        "适用": "统一我们的标注格式，便于后续扩展",
    },
}

print("项目数据策略路线图")
print("=" * 60)
for step, info in strategy.items():
    print(f"\n{step}: {info['动作']}")
    for k, v in info.items():
        if k != '动作':
            print(f"  {k}: {v}")

## 结论

- **考古 ML 领域数据集极度碎片化**，不存在像 ImageNet 那样的标准 benchmark
- **实际做法**: 几乎所有团队都用 ImageNet/COCO 预训练 + 自建小规模数据集
- **对我们的启示**: 不要等待"完美数据集"，先用手头数据 + 迁移学习做起来
- **Roboflow** 可以作为标注工具，降低标注门槛
- **Archaeoscape 的标注规范** 可以作为参考标准

---

## 行动计划：下一步（2026-07-10）

以下是按优先级排列的下一步行动。每完成一项打勾，有突破就在此 notebook 记录。

### 第一层：环境与基础设施 ✅ 已完成

| # | 任务 | 说明 | 状态 |
|---|------|------|------|
| 1 | VPS 安装 Python 工具链 | uv 0.11.28 + venv | ✅ |
| 2 | VPS 安装 QGIS | QGIS 3.34.4-Prizren | ✅ |
| 3 | VPS 安装项目依赖 | PyTorch 2.13.0, GeoPandas 1.1.4, scikit-learn 1.9.0 | ✅ |
| 4 | 本地同步项目到 VPS | GitHub 同步 | ✅ |

**验证**: `python -c "import torch, geopandas; print('OK')"` ✅

### 第二层：PDF 数据提取 ⏳ 进行中

| # | 任务 | 说明 | 状态 |
|---|------|------|------|
| 5 | 下载 PDF 报告 | R2 考古期刊 → VPS data/pdfs/ | ✅ 13份已下载 (723MB) |
| 6 | 提取墓葬结构化数据 | Baidu OCR → 考古信息提取 → JSON | ⏳ 满城汉墓已跑通，正则需优化 |
| 7 | 数据质量校验 | 检查墓口长/宽/深数值合理性 | ⬜ |
| 8 | 保存为 CSV/JSON | 存入 data/ 目录 | ⬜ |

**已下载的 PDF**:
- 满城汉墓(上下)、马王堆(上下)、广州汉墓(3版本)、殷虚妇好墓
- 江陵雨台山楚墓、青海柳湾、信阳楚墓、前蜀王建墓、北京大葆台汉墓

**R2 桶中还有**: 文物期刊 117份 (1950-1961)、考古学报 25份 (1936-1960) — 可按需下载

### 第三层：QGIS 可视化（数据到手后执行）

| # | 任务 | 说明 | 阻塞项 | 状态 |
|---|------|------|--------|------|
| 9 | 创建 QGIS 项目模板 | 设置 CRS（WGS84 / CGCS2000）、底图、图层结构 | #2 | ⬜ |
| 10 | 墓葬点位图 | 将提取的墓葬坐标导入为点图层，按年代/形制着色 | #8, #9 | ⬜ |
| 11 | 分布热力图 | 按年代分层制作墓葬分布图，观察空间聚集 | #10 | ⬜ |
| 12 | 器物类型空间分布 | 不同器物类型在空间上的分布差异 | #10 | ⬜ |

**成功标准**: 在 QGIS 中看到墓葬的空间分布图，能初步判断是否存在区域聚集模式。

**关键问题**（数据说话后再回答）:
- 不同时期墓葬的空间分布是否有明显偏移？
- 特定器物类型是否集中在某些区域？
- 墓葬形制的地理分布是否有规律？

### 第四层：PyTorch 图像分类 ✅ 基线跑通 + xunzhilu 聚类完成

| # | 任务 | 说明 | 状态 |
|---|------|------|------|
| 13 | 下载 ArchaeoMind 数据集 | Kaggle, 1345 张图片 (196MB) | ✅ |
| 14 | 下载 Persian Pottery 数据集 | GitHub, 170 张图片 (341MB) | ✅ |
| 15 | 搭建 PyTorch 分类脚本 | ResNet18 预训练 + 迁移学习 | ✅ |
| 16 | 在 ArchaeoMind 上测试 | **Test Acc: 80.3%** | ✅ |
| 17 | xunzhilu 墓葬照片聚类 | ResNet18 特征 + KMeans(k=8) | ✅ |
| 18 | xunzhilu 标注 + 分类训练 | 需用户视觉检查聚类结果后标注 | ⬜ 等待用户 |

**ArchaeoMind 结果**: ResNet18 FC层微调，5 epochs，CPU 3分钟
- artifact: precision 0.75, recall 0.92
- non_artifact: precision 0.90, recall 0.68

**xunzhilu 聚类结果** (543 imgs, 228 records):
- Cluster 0: 91 imgs / 89 records
- Cluster 1: 38 imgs / 16 records
- Cluster 2: 136 imgs / 37 records (最大)
- Cluster 3: 55 imgs / 7 records (集中)
- Cluster 4: 77 imgs / 67 records (分散)
- Cluster 5: 33 imgs / 2 records (极集中: rec-0019, rec-0077)
- Cluster 6: 35 imgs / 2 records (极集中: rec-0055, rec-0087)
- Cluster 7: 78 imgs / 8 records
- 可视化: `data/xunzhilu_clusters.html`

### 第五层：Baidu OCR 集成 ✅ 已跑通

| # | 任务 | 说明 | 状态 |
|---|------|------|------|
| 17 | 验证 Baidu OCR API 连通性 | API Key 有效 | ✅ |
| 18 | 构建 PDF→OCR→结构化 pipeline | `scripts/pdf_ocr_extract.py` | ✅ |
| 19 | 优化提取正则表达式 | 当前正则匹配率偏低 | ⏳ |

**已知限制**:
- 图片尺寸 ≤ ~2000px（用 DPI 100 渲染 PDF）
- 扫描版 PDF 无文本层，必须 OCR
- 满城汉墓已跑通，但正则提取墓葬细节的匹配率需优化

### 并行执行策略

以下任务**互不依赖，可以同时进行**：

```
第一层（环境） ──→ 第二层（PDF提取） ──→ 第三层（QGIS）
                  ↗
第四层（数据集下载 + PyTorch scaffold） ← 不依赖 PDF
第五层（Baidu OCR） ← 不依赖其他层
```

**今天可以先做的**:
1. VPS 环境搭建（第一层）
2. 下载 ArchaeoMind + Persian Pottery 数据集（第四层）
3. 验证 Baidu OCR 连通性（第五层）

**等用户输入才能做的**:
- PDF 报告 → 才能做第二层 → 才能做第三层

### 突破记录

以下记录每个关键突破的时间和发现。完成一项就在这里加一条。

In [ ]:
# 突破记录
breakthroughs = [
    {
        "时间": "2026-07-10",
        "突破": "VPS SSH key 配置完成，项目在 VPS 上成功 clone",
        "细节": "生成 ed25519 key (vps-kaogu)，添加到 GitHub account alexxhchen-nu",
    },
    {
        "时间": "2026-07-10",
        "突破": "考古 ML 数据集调研完成",
        "细节": "ArchaeoMind (Kaggle), Persian Pottery (GitHub), Archaeoscape (NeurIPS 2024) 已记录在 dataset-research.md",
    },
    {
        "时间": "2026-07-10",
        "突破": "行动计划拆解为 5 层 19 项任务",
        "细节": "环境→PDF→QGIS→PyTorch→OCR，可并行执行",
    },
    {
        "时间": "2026-07-10 11:55",
        "突破": "VPS 环境全部搭建完成",
        "细节": "PyTorch 2.13.0 + QGIS 3.34.4 + uv + venv",
    },
    {
        "时间": "2026-07-10 12:01",
        "突破": "3 个数据集全部下载完成",
        "细节": "ArchaeoMind (196MB, 1345张), Persian Pottery (341MB, 170张), Archaeoscape (9.2MB)",
    },
    {
        "时间": "2026-07-10 12:06",
        "突破": "PyTorch 分类模型首次训练成功",
        "细节": "ResNet18 迁移学习，5 epochs，Test Acc: 80.3%（超过 60% baseline 目标）",
    },
    {
        "时间": "2026-07-10 13:10",
        "突破": "R2 考古 PDF 批量下载完成",
        "细节": "13 份关键墓葬报告下载到 VPS data/pdfs/（723MB）：满城汉墓、马王堆、广州汉墓(3版本)、殷虚妇好墓、江陵雨台山楚墓、青海柳湾、信阳楚墓、前蜀王建墓、北京大葆台汉墓",
    },
    {
        "时间": "2026-07-10 13:19",
        "突破": "xunzhilu 墓葬照片聚类完成",
        "细节": "543 张图片（228 条墓葬记录）用 ResNet18 提取特征 + KMeans(k=8) 聚类。发现 8 个视觉群组，部分群组高度集中（如 Cluster 5/6 各仅 2 条记录）。输出: data/xunzhilu_clusters.json + HTML 可视化",
    },
    {
        "时间": "2026-07-10 13:25",
        "突破": "发现 xunzhilu-xushui 是 xunzhilu-v2 的完全子集",
        "细节": "317 个 xushui 文件与 v2 前 317 个文件完全相同。使用 v2 作为超集。",
    },
]

for b in breakthroughs:
    print(f"[{b['时间']}] {b['突破']}")
    print(f"  细节: {b['细节']}\n")


### 第六层：朝代识别主数据集建立 ✅

| # | 任务 | 说明 | 状态 |
|---|------|------|------|
| 19 | 建立 dynasty_dataset 目录结构 | 7朝代 × train/val/test，含 .gitkeep 占位 | ✅ |
| 20 | 迁入 Mendeley 明代陶瓷 | 100张 → 70/15/15 split，manifest.csv 初始化 | ✅ |
| 21 | 补充东汉/西汉/唐代数据 | 等 Bronze Ding 邮件回复 + Eaves Tiles 补充材料 | ⏳ |
| 22 | PDF 解析图像接入 | OCR 提取后按朝代落入对应目录，source=pdf_ocr | ⏳ |

**manifest.csv 字段**: `filename, split, dynasty, source, artifact_type, pdf_source, label_confidence`

**支持的 source 类型**: `mendeley` / `pdf_ocr` / `bronze_ding` / `eaves_tiles` / `manual`

In [ ]:
# 突破：dynasty_dataset 主数据集初始化
breakthroughs.append({
    "时间": "2026-07-11",
    "突破": "朝代识别主数据集 dynasty_dataset 建立",
    "细节": "7朝代（han_western/han_eastern/tang/song/yuan/ming/qing）× train/val/test 目录结构，空目录 .gitkeep 占位；Mendeley 明代 100 张按 70/15/15 迁入；manifest.csv 初始化含 source/artifact_type/pdf_source/label_confidence 字段，为多源接入做好准备",
})

b = breakthroughs[-1]
print(f"[{b['时间']}] {b['突破']}")
print(f"  细节: {b['细节']}")

[2026-07-11] 朝代识别主数据集 dynasty_dataset 建立
  细节: 7朝代（han_western/han_eastern/tang/song/yuan/ming/qing）× train/val/test 目录结构，空目录 .gitkeep 占位；Mendeley 明代 100 张按 70/15/15 迁入；manifest.csv 初始化含 source/artifact_type/pdf_source/label_confidence 字段，为多源接入做好准备